# 04 — DG38 Crop Preprocessing Ablation cho Cascade Detection -> Classification (Variants A–E)

**Mục tiêu (Giai đoạn 7 — Hạn mức đề xuất: 15 CU):**
- Giữ cố định **Detector (`detection/checkpoints_wastedetect/best_model.pt`)**, **Classifier (`checkpoints/dinov2_garbage_final.pth`)** và toàn bộ **`64,819` NMS proposals (`nms_iou=0.70`)** đã cache trong `evaluation_runs/val_grid_search_01/cache/predictions.sqlite3` trên **Clean Validation (`1,743` ảnh / `3,932` GT boxes)**.
- Tự động xác minh SHA-256 fingerprint của bộ cache gốc (`fingerprint.json`) trước khi chạy.
- So sánh 5 phương án tiền xử lý crop trước khi đưa vào DINOv2:
  - `A_original_crop`: Tight crop -> `Resize(256, 256)` -> `CenterCrop(224)` *(khớp bit-exact 100% với Validation gốc: `mAP@50 = 0.4399`, `76.63%` acc trên `3,663` matched GT boxes, `54.02%` `glass -> plastic`, `29.76%` `paper -> cardboard`)*.
  - `B_direct_resize`: Tight crop -> `Resize(224, 224)` (không cắt mất 23.4% diện tích viền).
  - `C_context_pad_15`: Mở rộng bbox thêm +15% mỗi cạnh -> `Resize(256, 256)` -> `CenterCrop(224)`.
  - `D_letterbox_224`: Tight crop -> Đệm vuông bảo toàn tỷ lệ khung hình `RGB(114, 114, 114)` -> `224x224`.
  - `E_context_pad_25_letterbox`: Mở rộng bbox thêm +25% mỗi cạnh -> Đệm vuông bảo toàn tỷ lệ khung hình -> `224x224`.
- Mỗi biến thể `B..E` được lưu trong bảng `variant_crops` của bản sao SQLite (`WORK_CACHE_DB`), **tuyệt đối không sửa đổi file `predictions.sqlite3` gốc hay kết quả Final Test cũ**.

In [ ]:
# [CELL 1] Thiết lập đường dẫn & kiểm tra SQLite Cache + Checkpoints của Clean Validation (1,743 ảnh)
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

DRIVE_ROOT = Path(os.environ.get("DG38_DRIVE_ROOT", "/content/drive/MyDrive/DG38_Colab_Package"))
REPO_ROOT = DRIVE_ROOT / "code" if (DRIVE_ROOT / "code" / "model.py").exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / "evaluation_runs" / "dg38_additional_experiments"
ARTIFACTS_DIR = (DRIVE_ROOT / "artifacts") if DRIVE_ROOT.exists() else EXP_DIR

SOURCE_CACHE_DB = REPO_ROOT / "evaluation_runs" / "val_grid_search_01" / "cache" / "predictions.sqlite3"
WORK_CACHE_DB = Path("/content/dg38_data/crop_ablation_predictions.sqlite3")
VALID_CLEAN_DIR = Path(os.environ.get("DG38_VALID_CLEAN_DIR", "/content/dg38_data/detection/WasteDetect_Dataset/valid_clean_candidate"))
VALID_COCO = VALID_CLEAN_DIR / "_annotations.coco.json"
DET_CKPT = REPO_ROOT / "detection" / "checkpoints_wastedetect" / "best_model.pt"
CLS_CKPT = REPO_ROOT / "checkpoints" / "dinov2_garbage_final.pth"

OBSERVED_CU_PER_HOUR = None      # <-- BẮT BUỘC: Nhập Usage Rate thực tế từ giao diện Colab (ví dụ: 1.48)
RUN_FULL_CROP_ABLATION = False   # Đổi thành True sau khi kiểm tra Pilot ở Cell 2 đạt APPROVED

if OBSERVED_CU_PER_HOUR is None or float(OBSERVED_CU_PER_HOUR) <= 0:
    raise ValueError("Vui lòng nhập OBSERVED_CU_PER_HOUR (Usage Rate từ 'Runtime -> View resources') trước khi chạy!")

print(f"SOURCE_CACHE_DB : {SOURCE_CACHE_DB} (exists={SOURCE_CACHE_DB.exists()})")
print(f"VALID_COCO      : {VALID_COCO} (exists={VALID_COCO.exists()})")
print(f"DET_CKPT        : {DET_CKPT} (exists={DET_CKPT.exists()})")
print(f"CLS_CKPT        : {CLS_CKPT} (exists={CLS_CKPT.exists()})")

In [ ]:
# [CELL 2] Chạy Pilot (đo tốc độ trên 25 ảnh) hoặc Toàn Bộ 5 Biến Thể Crop Preprocessing (A, B, C, D, E) trên Clean Validation
crop_script = EXP_DIR / "dg38_crop_ablation.py"
out_dir = ARTIFACTS_DIR / "runs" / "crop_ablation_val_clean"
mode = "run" if RUN_FULL_CROP_ABLATION else "pilot"

cmd = [
    sys.executable,
    str(crop_script),
    "--mode", mode,
    "--valid-coco", str(VALID_COCO),
    "--valid-images", str(VALID_CLEAN_DIR),
    "--det-ckpt", str(DET_CKPT),
    "--cls-ckpt", str(CLS_CKPT),
    "--base-sqlite", str(SOURCE_CACHE_DB),
    "--work-sqlite", str(WORK_CACHE_DB),
    "--output-dir", str(out_dir),
    "--cu-per-hour", str(OBSERVED_CU_PER_HOUR),
]
print(f"Đang chạy Crop Ablation ở chế độ --mode {mode} ...")
if VALID_COCO.exists() and SOURCE_CACHE_DB.exists() and CLS_CKPT.exists():
    res = subprocess.run(cmd, text=True)
    if res.returncode != 0:
        raise RuntimeError("Crop Ablation gặp lỗi.")
else:
    print("[INFO] Hãy đảm bảo VALID_COCO, SOURCE_CACHE_DB và CLS_CKPT đều tồn tại (chạy Notebook 00 trước).")